# Production Ranking Model — XGBoost `rank:ndcg`

Walks forward through quarterly cutoff dates, re-fitting a ranking model on
each expanding window, and evaluates the **top-of-the-list** metrics that map
to the screening product (top-10 / top-25 / top-quintile picks) instead of the
pooled regression metrics we already know are weak.

**Why `rank:ndcg` instead of regression:** the product asks "which names rank
highest", not "what will the return be". NDCG optimizes the head of the ranking
directly — in backtests it roughly doubled the top-10 excess return and
precision@10 vs. the regression baseline, at the cost of pooled rank IC.

**Purpose:** benchmark gate + production artifact. The last cell re-fits on all
history and saves the booster so the API/service layer can score any date cohort.


In [1]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from scipy.stats import spearmanr
from stockidence.storage import Warehouse

IT = Path.cwd()
if not (IT / "train_dataset_quarterly.parquet").exists():
    raise SystemExit("run this notebook from Model/ (or fix PARQUET path)")
REPO = IT.parent                       # warehouse DB lives at repo/data/
DB = REPO / "data" / "stockidence.duckdb"
GRAIN = "quarterly"
PARQUET = IT / f"train_dataset_{GRAIN}.parquet"
ARTIFACT = IT / "artifacts"
ARTIFACT.mkdir(exist_ok=True)

EXCLUDED = {"NBIS", "BRK.B", "AMBP", "CRM", "BE", "FI"}
MODEL_NAME = "ranking_ndcg"


### 1. Load the curated training universe

`build_dataset.py` already writes only the training-universe tickers to the
parquet (see `--tickers` / `--tickers-file`); this cell applies the final
exclusion list and drops rows the feature set literally cannot see.

In [2]:
NEW_PRICE_FEATURES = [
    "price_to_sma200", "stddev_252", "max_drawdown_252", "atr_pct",
    "return_3m", "return_12m", "distance_from_52wk_high"]
NEW_FUND_FEATURES = [
    "roe", "roa", "debt_equity", "current_ratio", "cash_to_assets", "fcf_to_assets",
    "roe_chg_qoq", "roa_chg_qoq", "fcf_to_assets_chg_qoq", "debt_equity_chg_qoq"]
CATEGORICAL = ["sector"]
FEATURES = NEW_PRICE_FEATURES + NEW_FUND_FEATURES + CATEGORICAL

df = pd.read_parquet(PARQUET)
TICKERS = sorted(set(df["ticker"].unique()) - EXCLUDED)
model_df = df[df["ticker"].isin(TICKERS)].dropna(subset=FEATURES + ["target_return"]).copy()
model_df["sector"] = model_df["sector"].astype("category")
print(f"universe: {len(TICKERS)} tickers | rows: {len(model_df)} | "
      f"{model_df['date'].min().date()} → {model_df['date'].max().date()}")


universe: 275 tickers | rows: 8478 | 2012-07-01 → 2026-04-01


### 2. Feature engineering — FE41

Same crossing/averaging recipe as the benchmark harness: raw macro momentum,
vol-scaled momentum, cross-sectional percent ranks (the order statistic the
ranking objective cares about) and market-relative (demeaned) momentum.

In [3]:
def engineer(d):
    d = d.copy().sort_values(["ticker", "date"])
    d["return_6m"] = d.groupby("ticker")["close"].pct_change(2)
    d["return_24m"] = d.groupby("ticker")["close"].pct_change(8)
    vol_mean = d["stddev_252"].mean()
    d["mom6_vol"] = d["return_6m"] / d["stddev_252"].clip(lower=1e-6) * vol_mean
    d["mom12_vol"] = d["return_12m"] / d["stddev_252"].clip(lower=1e-6) * vol_mean
    d["dist_52wk_vol"] = d["distance_from_52wk_high"] / d["stddev_252"].clip(lower=1e-6) * vol_mean
    for c in ["return_3m", "return_6m", "return_12m", "return_24m",
              "roe", "roa", "fcf_to_assets", "price_to_sma200", "distance_from_52wk_high",
              "atr_pct", "debt_equity", "max_drawdown_252"]:
        d[f"rk_{c}"] = d.groupby("date")[c].rank(pct=True)
    for c in ["return_3m", "return_6m", "return_12m", "return_24m"]:
        d[f"rel_{c}"] = d[c] - d.groupby("date")[c].transform("mean")
    return d

FE41 = [
    *NEW_PRICE_FEATURES, *NEW_FUND_FEATURES,
    "sma_200", "max_252",
    "return_6m", "return_24m",
    "mom6_vol", "mom12_vol", "dist_52wk_vol",
    *[f"rk_{c}" for c in ["return_3m", "return_6m", "return_12m", "return_24m",
        "roe", "roa", "fcf_to_assets", "price_to_sma200", "distance_from_52wk_high",
        "atr_pct", "debt_equity", "max_drawdown_252"]],
    *[f"rel_{c}" for c in ["return_3m", "return_6m", "return_12m", "return_24m"]],
]

md = engineer(model_df).copy()
md = md.dropna(subset=FE41 + ["sector"]).reset_index(drop=True)
md["sector"] = md["sector"].astype("category")
print(f"FE41: {len(FE41)} features | model rows: {len(md)} | "
      f"tickers: {md['ticker'].nunique()}")


FE41: 40 features | model rows: 6925 | tickers: 191


### 3. Walk-forward evaluation (expanding window, quarterly)

- Cutoff every quarter start from 2019 → 2025; train on `< cutoff`, test on
  `== cutoff` (one quarter of rows).
- Relevance grades = per-date quintile of realized forward return (0..4). Using
  grades on *train* rows only keeps the labels PIT and lets NDCG make full use
  of the order information.
- Group boundaries (`set_group`) tell XGBoost each date is a query group.

In [4]:
RANK_PARAMS = {
    "objective": "rank:ndcg", "tree_method": "hist",
    "max_depth": 3, "eta": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
    "reg_alpha": 0.1, "reg_lambda": 1.0, "seed": 42, "nthread": -1,
}
N_GRADES = 5
CUTOFFS = pd.date_range("2019-01-01", "2025-04-01", freq="QS")

def grades_for(rows, n=N_GRADES):
    return rows.groupby("date")["target_return"].transform(
        lambda s: pd.qcut(s.rank(method="first"), n, labels=False)).astype(int)

R = []
for c in CUTOFFS:
    tr = md["date"] < c
    te = md["date"] == c
    if te.sum() < 5:
        continue
    y = grades_for(md.loc[tr])
    qid = md.loc[tr, "date"].astype("category").cat.codes.values
    dtr = xgb.DMatrix(md.loc[tr, FE41], label=y)
    dtr.set_group(np.bincount(qid))
    bst = xgb.train(RANK_PARAMS, dtr, num_boost_round=150)
    p = bst.predict(xgb.DMatrix(md.loc[te, FE41]))
    R.append(pd.DataFrame({"date": md.loc[te, "date"].values,
                           "ticker": md.loc[te, "ticker"].values,
                           "pred": p, "real": md.loc[te, "target_return"].values}))
R = pd.concat(R, ignore_index=True)
print(f"test rows: {len(R)}  quarters: {R['date'].nunique()}")


test rows: 3912  quarters: 26


### 4. Evaluation — head-of-ranking metrics

These are the numbers the screener is responsible for. `excess` is *market
relative*: each quarter, the top-K equal-weight mean minus that quarter's
equal-weight universe mean, so bull-market drift is removed. `precision@K` is
overlap between predicted top-K and realized top-K (random = K / n).

In [5]:
def head_metrics(R):
    rows = []
    for d, g in R.groupby("date"):
        g = g.sort_values("pred", ascending=False).reset_index(drop=True)
        uni = g["real"].mean()
        rr = g["real"].rank(ascending=False)
        rows.append({
            "date": d, "n": len(g), "uni": uni,
            "top10_exc": g.head(10)["real"].mean() - uni,
            "top25_exc": g.head(25)["real"].mean() - uni,
            "topQ_exc": g.head(len(g) // 5)["real"].mean() - uni,
            "prec10": rr.head(10).le(10).mean(),
            "prec25": rr.head(25).le(25).mean(),
        })
    return pd.DataFrame(rows)

def show(col):
    s = E[col]
    t = s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
    print(f"  {col:10s} {s.mean()*100:+.2f} pp/qtr  t={t:+.2f}  "
          f"{100*(s > 0).mean():.0f}% of quarters")

ic = spearmanr(R["real"], R["pred"]).statistic
E = head_metrics(R)
print(f"rank IC (pooled): {ic:+.4f}   (bench random = 0; regression ~ +0.12)")
show("top10_exc"); show("top25_exc"); show("topQ_exc")
print(f"  precision@10: {E['prec10'].mean()*100:.1f}%   (random={10/E['n'].mean()*100:.1f}%)")
print(f"  precision@25: {E['prec25'].mean()*100:.1f}%   (random={25/E['n'].mean()*100:.1f}%)")
print()
R["year"] = R["date"].dt.year
for y, g in R.groupby("year"):
    sub = head_metrics(g)
    print(f"  {y}: top10 {sub['top10_exc'].mean()*100:+.2f} pp/qtr | "
          f"top25 {sub['top25_exc'].mean()*100:+.2f} | "
          f"topQ {sub['topQ_exc'].mean()*100:+.2f}  ({len(sub)} qtrs)")


rank IC (pooled): +0.0863   (bench random = 0; regression ~ +0.12)
  top10_exc  +3.27 pp/qtr  t=+1.39  62% of quarters
  top25_exc  +3.37 pp/qtr  t=+2.09  65% of quarters
  topQ_exc   +3.54 pp/qtr  t=+2.44  73% of quarters
  precision@10: 19.2%   (random=6.6%)
  precision@25: 30.5%   (random=16.6%)

  2019: top10 +3.38 pp/qtr | top25 +2.07 | topQ +1.82  (4 qtrs)
  2020: top10 +14.85 pp/qtr | top25 +11.53 | topQ +12.65  (4 qtrs)
  2021: top10 -3.18 pp/qtr | top25 -0.04 | topQ +0.00  (4 qtrs)
  2022: top10 +3.35 pp/qtr | top25 +1.22 | topQ +1.11  (4 qtrs)
  2023: top10 +5.20 pp/qtr | top25 +4.64 | topQ +3.98  (4 qtrs)
  2024: top10 -2.71 pp/qtr | top25 +0.72 | topQ +1.28  (4 qtrs)
  2025: top10 +0.78 pp/qtr | top25 +3.54 | topQ +4.42  (2 qtrs)


### 5. Benchmark: top-K vs the S&P 500

The universe already beats the S&P by design (equal-weight vs cap-weight). What
we care about is whether the *model's* top-K adds excess on top of both the
universe and the index. SPX forward quarter returns come from the FRED market
series, labeled as-of each test date.

In [6]:
def spx_forward(R):
    wh = Warehouse(DB)
    with wh.connect(read_only=True) as con:
        spx = pd.read_sql("SELECT date, spx FROM mart.m_fred_market "
                          "WHERE spx IS NOT NULL ORDER BY date", con)
    spx["date"] = pd.to_datetime(spx["date"])
    spx["q"] = spx["date"].dt.to_period("Q").dt.start_time.dt.normalize()
    qfwd = spx.groupby("q")["spx"].last().sort_index().pct_change().shift(-1)
    return R.assign(spx_fwd=R["date"].map(qfwd))

R2 = spx_forward(R).dropna(subset=["spx_fwd"])
rows = []
for d, g in R2.groupby("date"):
    g = g.sort_values("pred", ascending=False)
    rows.append({"date": d,
                 "top20": g.head(20)["real"].mean(),
                 "spx": g["spx_fwd"].iloc[0],
                 "uni": g["real"].mean()})
S = pd.DataFrame(rows)
def summ(col):
    v = S[col]
    t = v.mean() / (v.std(ddof=1) / np.sqrt(len(v)))
    return v.mean() * 100, t, 100 * (v > 0).mean()
m, t, h = summ("top20")
sx, st, sh = summ("spx")
ux, ut, uh = summ("uni")
print(f"quarters w/ SPX benchmark: {len(S)}")
print(f"  top-20 mean        : {m:+.2f}% / qtr")
print(f"  S&P 500 mean       : {sx:+.2f}% / qtr  (t={st:+.2f})")
print(f"  universe mean      : {ux:+.2f}% / qtr  (t={ut:+.2f})")
print(f"  top-20 vs S&P      : {m - sx:+0.2f} pp/qtr, beats S&P {h:.0f}% of quarters")
S["year"] = S["date"].dt.year
print()
for y, g in S.groupby("year"):
    v = g["top20"] - g["spx"]
    print(f"  {y}: top-20 vs S&P {v.mean()*100:+.2f} pp/qtr "
          f"({100*(v > 0).mean():.0f}%)")


quarters w/ SPX benchmark: 26
  top-20 mean        : +7.50% / qtr
  S&P 500 mean       : +3.74% / qtr  (t=+2.17)
  universe mean      : +4.06% / qtr  (t=+2.24)
  top-20 vs S&P      : +3.76 pp/qtr, beats S&P 73% of quarters

  2019: top-20 vs S&P +1.34 pp/qtr (50%)
  2020: top-20 vs S&P +15.93 pp/qtr (75%)
  2021: top-20 vs S&P -1.23 pp/qtr (50%)
  2022: top-20 vs S&P +4.32 pp/qtr (75%)
  2023: top-20 vs S&P +4.85 pp/qtr (50%)
  2024: top-20 vs S&P -0.53 pp/qtr (50%)
  2025: top-20 vs S&P -0.49 pp/qtr (50%)


/var/folders/jh/dwpc5mps01v9yszs4yfj4qn80000gn/T/ipykernel_87401/1243143142.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  spx = pd.read_sql("SELECT date, spx FROM mart.m_fred_market "


### 6. Production artifact

Re-fit on **all** history and save the booster + metadata so the service layer
can score requests without re-training. Grading uses every date's realized
returns — those are *outcomes*, not features, and the walk-forward above already
measured generalization; this fit is the deployed version.


In [7]:
yticks = grades_for(md)
qid = md["date"].astype("category").cat.codes.values
dm = xgb.DMatrix(md[FE41], label=yticks)
dm.set_group(np.bincount(qid))
bst = xgb.train(RANK_PARAMS, dm, num_boost_round=150)

meta = {
    "model": MODEL_NAME,
    "objective": "rank:ndcg",
    "grain": GRAIN,
    "features": FE41,
    "excluded": sorted(EXCLUDED),
    "n_rounds": 150,
    "params": RANK_PARAMS,
    "n_tickers": int(md["ticker"].nunique()),
    "n_rows": int(len(md)),
    "date_range": [str(md["date"].min().date()), str(md["date"].max().date())],
}
bst.save_model(str(ARTIFACT / f"{MODEL_NAME}.json"))
with open(ARTIFACT / f"{MODEL_NAME}.meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
print(f"saved {ARTIFACT / MODEL_NAME}.json + .meta.json")
print(f"top features by gain: {', '.join(k for k,_ in sorted(bst.get_score(importance_type='gain').items(), key=lambda x:-x[1])[:10])}")


saved /Users/nativeongfuel/Stockidence/Model/artifacts/ranking_ndcg.json + .meta.json
top features by gain: max_drawdown_252, rk_max_drawdown_252, rk_atr_pct, atr_pct, distance_from_52wk_high, return_12m, rel_return_24m, return_24m, mom6_vol, rel_return_12m


### 7. Sanity: load artifact, score the latest quarter

Proves the artifact round-trips and gives a peek at what the live screener
would emit for the most recent cohort.


In [8]:
m2 = xgb.Booster()
m2.load_model(str(ARTIFACT / f"{MODEL_NAME}.json"))
last = md["date"] == md["date"].max()
Xl = xgb.DMatrix(md.loc[last, FE41])
sc = m2.predict(Xl)
top = md.loc[last, ["ticker", "sector"]].assign(score=sc) \
            .sort_values("score", ascending=False).head(20)
top['rank'] = range(1, len(top)+1)
print(f"as of {md.loc[last, 'date'].iloc[0].date()} — top-20 cohort:")
print(top[["rank", "ticker", "sector", "score"]].to_string(index=False))


as of 2026-04-01 — top-20 cohort:
 rank ticker                 sector    score
    1   TEAM             Technology 1.525705
    2    ACN             Technology 1.488374
    3     ZS             Technology 1.420627
    4    NOW             Technology 1.408422
    5   INTU             Technology 1.112824
    6   WDAY             Technology 1.076249
    7    ZTS             Healthcare 0.911958
    8   DKNG Consumer Discretionary 0.773452
    9   ADBE             Technology 0.632423
   10    BSX             Healthcare 0.587547
   11   MRNA             Healthcare 0.497873
   12   PYPL             Financials 0.420120
   13    CMG Consumer Discretionary 0.417009
   14    NKE Consumer Discretionary 0.380093
   15     ON             Technology 0.254180
   16   SNOW             Technology 0.235405
   17   REGN             Healthcare 0.224526
   18   TSCO Consumer Discretionary 0.193524
   19   DXCM             Healthcare 0.187780
   20    HAL                 Energy 0.179841
